# Airbnb - Stays Host Communication Response Time Performance

In [1]:
import pandas as pd  
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_guest = pd.read_csv('../Data/009/fct_guest_inquiries.csv', parse_dates=['inquiry_date'])

pl_guest = pl.read_csv('../Data/009/fct_guest_inquiries.csv', try_parse_dates= True)

# Pregunta 1

### ¿Cuál es el tiempo mínimo de respuesta de los anfitriones (en horas) para las consultas de los huéspedes en enero de 2024? Esta métrica ayudará a identificar si algún anfitrión está estableciendo un estándar de respuesta excepcionalmente rápido.

```SQL
SELECT
    MIN(response_time_hours) min_response
FROM fct_guest_inquiries
WHERE ((EXTRACT(MONTH FROM inquiry_date) = 1) AND
       (EXTRACT(YEAR FROM inquiry_date) = 2024));
```

In [4]:
enero = df_guest[
    (df_guest['inquiry_date'].dt.month == 1) &
    (df_guest['inquiry_date'].dt.year == 2024)
].reset_index()

res = enero.agg(
    min_response = ('response_time_hours','min')
)

In [6]:
res = pl_guest.filter(
    (pl.col('inquiry_date').dt.month() == 1) &
    (pl.col('inquiry_date').dt.year() == 2024)
).select(
    pl.col('response_time_hours').min().alias('min_response')
)

# Pregunta 2

### Para las consultas de los huéspedes realizadas en enero de 2024, ¿cuál es el tiempo promedio de respuesta de los anfitriones redondeado a la hora más cercana? Este promedio proporcionará información sobre la capacidad de respuesta general de los anfitriones.

```SQL
SELECT
    ROUND(AVG(response_time_hours)::NUMERIC,0) AS avg_time_response
FROM fct_guest_inquiries
WHERE ((EXTRACT(MONTH FROM inquiry_date) = 1) AND
       (EXTRACT(YEAR FROM inquiry_date) = 2024));
```

In [24]:
enero = df_guest[
    (df_guest['inquiry_date'].dt.month == 1) &
    (df_guest['inquiry_date'].dt.year == 2024)
].reset_index()

res = enero.agg(
    avg_time_response = ('response_time_hours', 'mean')
).round(0).T

In [25]:
res = pl_guest.filter(
    (pl.col('inquiry_date').dt.month() == 1) &
    (pl.col('inquiry_date').dt.year() == 2024)
).select(
    pl.col('response_time_hours').mean().round(0).alias('avg_tiem_response')
)

# Pregunta 3

### Enumera el inquiry_id y el response_time_hours para las consultas de los huéspedes realizadas entre el 16 y el 31 de enero de 2024 que tardaron más de 2 horas en ser respondidas. Este desglose ayudará a identificar con precisión a los anfitriones con tiempos de respuesta más lentos.

```SQL
SELECT
    inquiry_id,
    response_time_hours
FROM fct_guest_inquiries
WHERE (inquiry_date BETWEEN '2024-01-16' AND '2024-01-31') AND
      response_time_hours > 2;
```

In [29]:
w3w4 = df_guest[
    (df_guest['inquiry_date'].between('2024-01-16','2024-01-31')) &
    (df_guest['response_time_hours'] > 2)
].reset_index()

res = w3w4[['inquiry_id', 'response_time_hours']]

In [30]:
res = pl_guest.filter(
    (pl.col('inquiry_date').is_between(date(2024,1,16),date(2024,1,31)))&
    (pl.col('response_time_hours') > 2)
).select([
    'inquiry_id', 'response_time_hours'
]
)

res

inquiry_id,response_time_hours
i64,f64
4,3.0
6,2.1
7,4.0
9,3.5
10,2.5
14,2.3
